In [ ]:
# Load packages
from functools import reduce
import numpy as np 
import pandas as pd 
import os 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Load packages 
import os
import pandas as pd
import gcsfs
import ast

# Define input and output directories within the All of Us Researcher Workbench
input_path = f'{os.environ["WORKSPACE_BUCKET"]}/data/hail/'
output_path = f'{os.environ["WORKSPACE_BUCKET"]}/data/chip'

# Initialize Google Cloud Storage filesystem
fs = gcsfs.GCSFileSystem()

# List all files in the input directory
files = fs.ls(input_path)

# Initialize counters to track PASS and failing entries across all files
total_pass = 0
total_fail = 0

# Store dataframes containing variants that fail quality-control thresholds
dfs = [] 

# Iterate through all files in the input directory
for f in files:
    # Process only compressed TSV files
    if f.endswith('.tsv.bgz'):

        print(f"Processing: {f}")

        # Initialize counters for the current file
        file_pass = 0
        file_fail = 0

        # Open the file directly from Google Cloud Storage
        with fs.open(f, 'rb') as fh:
            # Read the file in chunks to avoid loading the entire dataset into memory
            for df in pd.read_csv(
                fh,
                sep='\t',
                compression='gzip',
                chunksize=1000
            ):
                # Retain only entries that passed the variant-level quality filters (FT = PASS)
                df = df[df['FT'] == 'PASS'].copy()
                
                # Skip chunks without any PASS entries
                if df.empty:
                    continue
                    
                # Convert the alleles column from string representation to Python lists
                df["alleles"] = df["alleles"].apply(ast.literal_eval)

                # Extract the reference and alternative alleles
                df["ref"] = df["alleles"].str[0]
                df["alt"] = df["alleles"].str[1]
                
                # Construct a unique variant identifier using chromosome position,
                # reference allele, and alternative allele
                df['id'] = df['locus'] + ':' + df['ref'] + ':' + df['alt']
            
                # Identify variants failing either read-depth (DP) or allele-balance (AB)
                # thresholds:
                #   DP < 20: insufficient read depth
                #   AB < 0.35: allele balance below the accepted range
                #   AB > 0.65: allele balance above the accepted range
                fail_mask = (
                    (df['DP'] < 20) |
                    (df['AB'] < 0.35) | 
                    (df['AB'] > 0.65)
                )
                
                # Append entries failing either threshold to the results list
                dfs.append(df[fail_mask]) 
                
                # Validate that retained entries satisfy the allele-balance thresholds
                if df[~fail_mask]['AB'].min() < 0.35: 
                    print(df[~fail_mask]['AB'].min()) 
                    break 
                    
                if df[~fail_mask]['AB'].max() > 0.65: 
                    print(df[~fail_mask]['AB'].max()) 
                    break 
                
                # Validate that retained entries satisfy the read-depth threshold
                if df[~fail_mask]['DP'].min() < 20: 
                    print(df[~fail_mask]['AB'].max()) 
                    break 
                
                # Count the number of entries failing at least one quality-control threshold
                count = fail_mask.sum()

                # Update the total number of PASS entries and failing entries for this file
                file_pass += len(df)
                file_fail += count
                
                # Confirm that the number of identified failing entries matches the failure count
                if not (len(df[fail_mask]) == count):
                    print("Mismatch between fail count") 
                    break 
        
        # Update overall counters with results from the current file
        total_pass += file_pass
        total_fail += file_fail

        # Report quality-control results for the current file
        print(f"PASS entries: {file_pass:,}")
        print(f"Failing entries: {file_fail:,}")
        print(
            f"Failing proportion: "
            f"{file_fail / file_pass:.4%}"
            if file_pass else "No PASS entries"
        )

# Report aggregate quality-control results across all files
print("\n=== Overall results ===")
print(f"Total PASS entries: {total_pass:,}")
print(f"Total failing entries: {total_fail:,}")
print(
    f"Overall failing proportion: {total_fail / total_pass:.4%}"
    if total_pass else "No PASS entries"
)

# Concatenate all dataframes containing entries that failed quality-control thresholds
failed = pd.concat(dfs)

# Rename columns to identify the variant and corresponding sample
failed.rename(columns={'id':'variant',
                       's':'sample'}, 
              inplace=True)

# Verify that the number of exported failing entries matches the overall failure count
print(f"Check mistamch failing entries: {(len(failed) == total_fail)}")

# Define local and Google Cloud Storage output file paths
local_file = f"{os.getcwd()}/remove.txt"
gcs_file = f"{output_path}/remove.txt"

# Export the failing variant and sample identifiers as a tab-delimited text file
# This file can subsequently be used to remove the identified variant-sample combinations
failed[['variant', 'sample']].to_csv(
    local_file,
    sep='\t',
    index=False
)

# Upload the generated removal list to the All of Us workspace bucket
subprocess.run(
    ["gsutil", "cp", local_file, gcs_file],
    check=True
)

print(f"Uploaded: {gcs_file}")